# 示例: 运行耦合的水文-水力模型

**脚本:** `examples/run_coupled_model.py`

## 目的
此示例用于演示如何使用`SimulationController`来构建和运行一个简单的耦合模型。这是框架的核心功能之一，允许将不同类型的模型组件（如水文模型、水力模型）连接成一个统一的模拟网络。

此笔记本将展示如何：
1.  分别创建水文（`HydrologicalModel`）和水力（`HydraulicModel`）模型组件的实例。
2.  使用`SimulationController`将这些组件添加到一个网络中。
3.  将水文模型的出流连接到水力模型的入流。
4.  定义全局输入（如降雨），并运行整个耦合模拟。
5.  检查并可视化最终的模拟结果。

## 1. 环境设置

首先，我们导入所需的框架组件和模型组件。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

# 框架组件
from common.controller import SimulationController

# 水文模型组件
from hydro_model.model import HydrologicalModel
from hydro_model.runoff import SCSCurveNumberModule
from hydro_model.routing import SimpleRouting

# 水力模型组件
from preissmann_model.model import HydraulicModel
from preissmann_model.reach import RiverReach
from preissmann_model.cross_section import RectangularCrossSection

## 2. 创建模型组件

我们创建两个独立的模型组件：
- **一个水文流域 (`catchment`)**: 它使用SCS方法根据降雨产生径流。
- **一个水力河道 (`river`)**: 它使用Preissmann一维水动力模型来演算河道中的洪水波。

In [ ]:
# 水文组件
runoff_mod = SCSCurveNumberModule(CN=75)
routing_mod = SimpleRouting(k_q=0.8, k_s=0.2)
catchment = HydrologicalModel(
    name="C1_main_catchment",
    runoff_module=runoff_mod,
    routing_module=routing_mod
)
print(f"创建水文组件: '{catchment.name}'")

# 水力组件
num_nodes = 11
reach_geom = RiverReach(
    cross_sections=[RectangularCrossSection(width=15) for _ in range(num_nodes)],
    lengths=np.full(num_nodes - 1, 500.0),
    slope=0.002,
    manning_n=0.035
)
downstream_level = 1.5 # meters
river = HydraulicModel(
    name="R1_main_river",
    reach=reach_geom,
    dt=60.0,
    downstream_level=downstream_level
)
river.Q[:] = 0.1
river.Z[:] = downstream_level
print(f"创建水力组件: '{river.name}'")

## 3. 构建和连接网络

我们创建一个`SimulationController`的实例，并将上面创建的两个组件添加到控制器中。然后，我们使用`connect`方法，将流域`C1_main_catchment`的出口连接到河流`R1_main_river`的入口。控制器会自动处理它们之间的数据传递。

In [ ]:
controller = SimulationController()
controller.add_component(catchment)
controller.add_component(river)

controller.connect("C1_main_catchment", "R1_main_river")
print("连接 'C1_main_catchment' -> 'R1_main_river'")

## 4. 定义和运行模拟

我们定义一个简单的降雨过程作为全局输入，然后调用控制器的`run`方法来执行整个耦合模拟。为了获取历史记录，我们需要迭代`run`方法返回的生成器。

In [ ]:
num_steps = 120
dt_controller = 60.0

rainfall_hydrograph = np.zeros(num_steps)
rainfall_hydrograph[10:40] = 15.0 # 15 mm/hr

global_inputs = {
    'rainfall': rainfall_hydrograph,
    'pet': np.full(num_steps, 0.1)
}

# 运行模拟并记录历史数据
history = []
history_generator = controller.run(
    num_steps=num_steps,
    dt=dt_controller,
    global_inputs=global_inputs
)
for status in history_generator:
    current_step_history = {}
    for name, comp in controller.components.items():
        current_step_history[name] = {'outflow': comp.get_outflow()}
    history.append(current_step_history)

print("\n模拟完成。")

## 5. 检查并可视化结果

模拟结束后，我们可以从`history`列表中提取每个组件在每个时间步的出流数据，并进行可视化。下图展示了：
- 蓝线：水文流域（`C1`）产生的径流过程线。
- 红线：该径流过程线输入到水力河道（`R1`）后，在河道末端演算出的出流过程线。可以看到明显的洪峰削减和延迟效果。

In [ ]:
catchment_outflow_history = [h['C1_main_catchment']['outflow'] for h in history]
river_outflow_history = [h['R1_main_river']['outflow'] for h in history]
timesteps = np.arange(num_steps) * dt_controller / 3600 # hours

plt.figure(figsize=(12, 6))
plt.plot(timesteps, catchment_outflow_history, 'b-o', label='Catchment Runoff (Input to River)')
plt.plot(timesteps, river_outflow_history, 'r-s', label='River Outflow (Routed)')

plt.title('Coupled Model Simulation: Catchment Runoff and River Routing')
plt.xlabel('Time (hours)')
plt.ylabel('Flow (m³/s)')
plt.legend()
plt.grid(True)
plt.show()